# 03 — Reading IWFM Output Files

IWFM writes results as HDF5 and text files. This notebook covers the
DLL-free readers for all of them:

- budget HDF files (`GW.hdf`, `StrmBud.hdf`, …)
- all-node groundwater heads (`GWHeadAll.hdf`)
- hydrograph HDF files (`GWHyd.hdf`, `StrmHyd.hdf`, `Subsidence.hdf`)
- zone-budget HDF files + zone definition files
- text outputs: hydrographs, final states, `.bud` budget tables,
  face flows and velocities
- `collect_budgets` — one long-form DataFrame across runs and budget
  types, ready for aggregation and comparison

**Requires:** the sample model with its `Results/` folder.

In [1]:
import os
from pathlib import Path


def find_sample_model():
    env = os.environ.get("IWFM_SAMPLE_MODEL")
    if env:
        return Path(env)
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / ".assets" / "sample_model"
        if cand.is_dir():
            return cand
    raise FileNotFoundError("sample model not found — see notebook 01")


SAMPLE_MODEL = find_sample_model()
RESULTS_DIR = SAMPLE_MODEL / "Results"
BUDGET_DIR = SAMPLE_MODEL / "Budget"
ZBUDGET_DIR = SAMPLE_MODEL / "ZBudget"

## Budget HDF files

`read_budget_hdf` returns `{"locations": [...], "data": {loc: DataFrame}}`
— one DataFrame per location, one column per budget component, with a
`DatetimeIndex`.

In [2]:
from iwfm_io import read_budget_hdf

gw_bud = read_budget_hdf(RESULTS_DIR / "GW.hdf")
print("locations:", gw_bud["locations"])
df = gw_bud["data"]["ENTIRE MODEL AREA"]
print(f"shape {df.shape}, {df.index[0].date()} - {df.index[-1].date()}")
df.head(3)

locations: ['Region1 (SR1)', 'Region2 (SR2)', 'ENTIRE MODEL AREA']
shape (3653, 17), 1990-10-02 - 2000-10-01


,Percolation,Beginning Storage (+),Ending Storage (-),Deep Percolation (+),Gain from Stream (+),Recharge (+),Gain from Lake (+),Boundary Inflow (+),Subsidence (+),Subsurface Irrigation (+),Tile Drain Outflow (-),Pumping (-),GW Return Flow (-),Outflow to Root Zone (-),Net Subsurface Inflow (+),Discrepancy (=),Cumulative Subsidence
datetime,,,,,,,,,,,,,,,,,
1990-10-02,1.414547e+07,1.640838e+12,1.640645e+12,0.0,-2.853643e+08,0.0,1.563706e+06,1.656614e+07,7.501907e+07,0.0,9012.502298,562064.516129,0.0,0.0,0.0,-3.706011,7.501907e+07
1990-10-03,1.415094e+07,1.640645e+12,1.640529e+12,0.0,-1.417441e+08,0.0,4.333493e+06,1.565458e+07,6.556101e+06,0.0,9736.568583,562064.516129,0.0,0.0,0.0,-0.921898,8.157517e+07
1990-10-04,1.415715e+07,1.640529e+12,1.640494e+12,0.0,-5.578854e+07,0.0,4.912072e+06,1.563629e+07,1.122334e+06,0.0,9813.609715,562064.516129,0.0,0.0,0.0,11.573208,8.269750e+07


In [3]:
# Works for every budget type IWFM produces
for fname in ["StrmBud.hdf", "RootZone.hdf", "LWU.hdf", "LakeBud.hdf",
              "UnsatZoneBud.hdf", "SWShed.hdf", "StrmNodeBud.hdf"]:
    path = RESULTS_DIR / fname
    if path.exists():
        r = read_budget_hdf(path)
        print(f"{fname:18s} locations: {r['locations']}")

StrmBud.hdf        locations: ['Reach1(REACH 1)', 'Reach2(REACH 2)', 'Reach3(REACH 3)']
RootZone.hdf       locations: ['Region1 (SR1)', 'Region2 (SR2)', 'ENTIRE MODEL AREA']
LWU.hdf            locations: ['Region1 (SR1)', 'Region2 (SR2)', 'ENTIRE MODEL AREA']
LakeBud.hdf        locations: ['Lake1']
UnsatZoneBud.hdf   locations: ['Region1 (SR1)', 'Region2 (SR2)', 'ENTIRE MODEL AREA']
SWShed.hdf         locations: ['WATERSHED 1', 'WATERSHED 2', 'WATERSHED 3']
StrmNodeBud.hdf    locations: ['NODE 1', 'NODE 8', 'NODE 19']


## Groundwater heads at all nodes

Passing `n_nodes`/`n_layers` labels the columns `node_N_layer_M`.

In [4]:
from iwfm_io import read_head_hdf

heads = read_head_hdf(RESULTS_DIR / "GWHeadAll.hdf", n_nodes=441, n_layers=2)
print(f"shape {heads.shape}, {heads.index[0].date()} - {heads.index[-1].date()}")
layer1 = [c for c in heads.columns if c.endswith("_layer_1")]
heads[layer1[:4]].head(3)

shape (3654, 882), 1990-10-01 - 2000-10-01


,node_1_layer_1,node_2_layer_1,node_3_layer_1,node_4_layer_1
datetime,,,,
1990-10-01,290.0,280.0000,280.0000,280.0000
1990-10-02,290.0,280.0403,280.0229,280.0229
1990-10-03,290.0,280.0585,280.0237,280.0235


## Hydrograph HDF files

One column per hydrograph print site (the sites are defined in the GW
main / stream main input files — see notebook 08 for linking them to
real-world wells and gauges).

In [5]:
from iwfm_io import read_hydrograph_hdf

for fname in ("GWHyd.hdf", "StrmHyd.hdf", "Subsidence.hdf",
              "TileDrainFlows.hdf"):
    path = RESULTS_DIR / fname
    if path.exists():
        df = read_hydrograph_hdf(path)
        print(f"{fname:20s} {df.shape}  "
              f"({df.index[0].date()} - {df.index[-1].date()})")

GWHyd.hdf            (3654, 42)  (1990-10-01 - 2000-10-01)
StrmHyd.hdf          (3653, 23)  (1990-10-02 - 2000-10-01)
Subsidence.hdf       (3654, 5)  (1990-10-01 - 2000-10-01)
TileDrainFlows.hdf   (3653, 6)  (1990-10-01 - 2000-09-30)


In [6]:
read_hydrograph_hdf(RESULTS_DIR / "GWHyd.hdf").iloc[:3, :5]

,col_1,col_2,col_3,col_4,col_5
datetime,,,,,
1990-10-01,280.000,280.000,280.000,280.000,280.000
1990-10-02,281.997,281.676,281.284,280.973,280.745
1990-10-03,283.971,283.330,282.547,281.926,281.471


## Zone budgets

Zone-budget HDF files hold per-element flows. Read raw, or pass a
`ZoneDefinition` (from a `ZoneDef_*.dat` file) to aggregate by zone.
These files are GB-scale on real models — the first read takes a while.

In [7]:
from iwfm_io import read_zbudget_hdf, read_zone_def

zpath = RESULTS_DIR / "GW_ZBud.hdf"
if zpath.exists():
    raw = read_zbudget_hdf(zpath)
    print("raw top-level keys:", list(raw.keys())[:6])

    zone_def_path = ZBUDGET_DIR / "ZoneDef_E1_E10.dat"
    if zone_def_path.exists():
        zdef = read_zone_def(zone_def_path)
        print(f"zone definition: extent='{zdef.extent}', zones {zdef.zones}")
        zoned = read_zbudget_hdf(zpath, zone_def=zdef)
        print("zoned keys:", list(zoned.keys())[:6])
else:
    print("GW_ZBud.hdf not present — run ZBudget to generate it")

raw top-level keys: ['metadata', 'data']
zone definition: extent='horizontal', zones {1: 'E1_E10'}
zoned keys: ['metadata', 'data', 'zones', 'face_flows']


## Text outputs

Everything IWFM prints as text has a reader too — useful because a fresh
executable run writes *text* outputs only (the `.hdf` hydrograph/head
files appear when the DLL opens the model in inquiry mode).

In [8]:
from iwfm_io import read_hydrograph_out

gwhyd = read_hydrograph_out(RESULTS_DIR / "GWHyd.out")
print(f"GWHyd.out: {gwhyd.shape}")
gwhyd.head(3)

GWHyd.out: (3654, 43)


,date,col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,...,col_33,col_34,col_35,col_36,col_37,col_38,col_39,col_40,col_41,col_42
0,09/30/1990_24:00,280.000,280.000,280.000,280.000,280.000,280.000,280.000,280.000,280.000,...,290.000,290.000,290.000,290.000,290.000,290.000,290.000,290.000,290.000,290.000
1,10/01/1990_24:00,281.997,281.676,281.284,280.973,280.745,281.142,280.877,280.674,280.519,...,279.513,278.838,278.256,277.452,276.797,276.172,275.587,275.041,274.531,273.746
2,10/02/1990_24:00,283.971,283.330,282.547,281.926,281.471,282.265,281.735,281.330,281.021,...,278.746,277.608,276.590,275.304,274.307,273.293,272.261,271.280,270.428,269.228


In [9]:
from iwfm_io import read_final_state_out

for fname in ("FinalGWHeads.out", "FinalSubsidence.out", "FinalLakeElev.out"):
    path = RESULTS_DIR / fname
    if path.exists():
        df = read_final_state_out(path)
        print(f"{fname:22s} {len(df)} rows  {df.columns.tolist()}")

FinalGWHeads.out       441 rows  ['ID', 'HP[1]', 'HP[2]']
FinalSubsidence.out    441 rows  ['ID', 'DC[1]', 'DC[2]', 'HC[1]', 'HC[2]']
FinalLakeElev.out      1 rows  ['ILAKE', 'HLAKE']


In [10]:
# Text .bud budget tables (produced by the Budget post-processor)
from iwfm_io import read_budget_text

for path in sorted(BUDGET_DIR.glob("*.bud")):
    sections = read_budget_text(path)
    print(f"{path.name:16s} {len(sections)} location sections")

DiverDetail.bud  5 location sections
GW.bud           3 location sections
Lake.bud         1 location sections
LWU.bud          3 location sections


LWU_TO.bud       3 location sections


RootZone.bud     3 location sections
RZBud_TO.bud     3 location sections
Strm.bud         3 location sections


StrmNode.bud     3 location sections
SWShed.bud       3 location sections
UnsatZone.bud    3 location sections


In [11]:
# Face flows and nodal velocities
from iwfm_io import read_flow_out, read_velocity_out

face = read_flow_out(RESULTS_DIR / "FaceFlow.out")
print(f"FaceFlow.out: {face.shape}")
vel_path = RESULTS_DIR / "GWVelocities.out"
if vel_path.exists():
    vel = read_velocity_out(vel_path)
    print(f"GWVelocities.out: {vel.shape}")

FaceFlow.out: (3653, 4)


GWVelocities.out: (3653, 8)


## One DataFrame across runs: `collect_budgets`

For comparison and aggregation work, `collect_budgets` reads any number
of budget files from any number of runs into a single **long-form**
DataFrame:

```
run       budget_type  location      datetime    component           value
baseline  GW           Region1 (SR1) 1990-10-02  Beginning Storage   1.2e8
```

Add more entries to `runs` (e.g. scenario Results folders from notebook
06) and every downstream groupby works across runs unchanged.

In [12]:
from iwfm_io import collect_budgets

runs = {"baseline": RESULTS_DIR}
budget_files = {"GW": "GW.hdf", "Stream": "StrmBud.hdf",
                "RootZone": "RootZone.hdf", "LWU": "LWU.hdf"}

long_df = collect_budgets(runs=runs, budget_files=budget_files)
print(f"{len(long_df):,} rows")
long_df.head(6)

1,216,449 rows


,run,budget_type,location,datetime,component,value
0,baseline,GW,Region1 (SR1),1990-10-02,Percolation,1.131816e+07
1,baseline,GW,Region1 (SR1),1990-10-03,Percolation,1.129179e+07
2,baseline,GW,Region1 (SR1),1990-10-04,Percolation,1.126613e+07
3,baseline,GW,Region1 (SR1),1990-10-05,Percolation,1.124116e+07
4,baseline,GW,Region1 (SR1),1990-10-06,Percolation,1.121687e+07
5,baseline,GW,Region1 (SR1),1990-10-07,Percolation,1.119326e+07


### Aggregating budgets correctly

Two conventions matter here. First, IWFM stamps each value at the
instant its period **ends** (`24:00` = next-day midnight), so naive
`.dt.year`/`.dt.month` labels drift at period boundaries — use
`water_year()` / `iwfm_day()` instead. Second, budget components have
different semantics: flows sum over a period, but the storage
**stocks** must not — `Beginning Storage` takes the period's first
value, `Ending Storage` and `Cumulative …` the last.
`aggregate_budget` applies both rules.

In [13]:
from iwfm_io import aggregate_budget

gw_entire = long_df[(long_df["budget_type"] == "GW")
                    & long_df["location"].str.contains("Entire", case=False)]
annual = (aggregate_budget(gw_entire, period="WY")
          .pivot_table(index="water_year", columns="component",
                       values="value"))
annual[["Beginning Storage (+)", "Ending Storage (-)",
        "Deep Percolation (+)", "Pumping (-)"]].head()

component,Beginning Storage (+),Ending Storage (-),Deep Percolation (+),Pumping (-)
water_year,,,,
1991,1.640838e+12,1.641221e+12,1.571540e+08,1.118820e+10
1992,1.641221e+12,1.643021e+12,1.068485e+09,1.117217e+10
1993,1.643021e+12,1.645383e+12,2.461421e+09,1.115623e+10
1994,1.645383e+12,1.650082e+12,6.036406e+09,1.114605e+10
1995,1.650082e+12,1.672327e+12,2.640340e+10,1.113929e+10


In [14]:
# Stock continuity is the tell that storage was handled right: each
# water year begins exactly where the previous one ended.
bool((annual["Beginning Storage (+)"].iloc[1:].values
      == annual["Ending Storage (-)"].iloc[:-1].values).all())

True

In [15]:
# Prefer pandas resample idioms? Ask for the frame with day_index=True:
# it comes back indexed by the *owning day* (iwfm_day), so calendar
# anchors label correctly — YE-SEP bins end on Sep 30 and read as the
# water year. Right for flow columns; storage stocks still want
# aggregate_budget.
from iwfm_io import open_model

gw_daily = open_model(SAMPLE_MODEL).budget_df(
    "GW", location="ENTIRE MODEL AREA", day_index=True)
gw_daily[["Deep Percolation (+)", "Pumping (-)"]].resample("YE-SEP").sum()

,Deep Percolation (+),Pumping (-)
datetime,,
1991-09-30,1.571540e+08,1.118820e+10
1992-09-30,1.068485e+09,1.117217e+10
1993-09-30,2.461421e+09,1.115623e+10
1994-09-30,6.036406e+09,1.114605e+10
1995-09-30,2.640340e+10,1.113929e+10
1996-09-30,7.202434e+10,1.112649e+10
1997-09-30,7.773930e+10,1.110845e+10
1998-09-30,6.525615e+10,1.108973e+10
1999-09-30,5.521810e+10,1.107054e+10


In [16]:
# Monthly seasonal means — iwfm_day() keeps month labels honest (a
# first-of-month midnight stamp belongs to the month it closes)
from iwfm_io import iwfm_day

seasonal = (gw_entire.assign(
                month=iwfm_day(gw_entire["datetime"]).dt.month)
            .groupby(["month", "component"], observed=True)["value"].mean()
            .unstack("component"))
seasonal.iloc[:, :4]

component,Beginning Storage (+),Boundary Inflow (+),Cumulative Subsidence,Deep Percolation (+)
month,,,,
1,1.720448e+12,-5.424579e+05,1.079983e+08,2.352365e+07
2,1.723234e+12,-6.275198e+05,1.072499e+08,4.805458e+07
3,1.724526e+12,-1.875410e+06,1.068767e+08,7.144744e+07
4,1.729267e+12,-7.287262e+06,1.051485e+08,2.708119e+08
5,1.736427e+12,-1.188373e+07,1.025427e+08,3.031296e+08
6,1.741892e+12,-8.873884e+06,1.007790e+08,1.344889e+08
7,1.744176e+12,-7.387019e+06,1.008538e+08,8.585654e+07
8,1.743658e+12,-6.833734e+06,1.031813e+08,7.226742e+07
9,1.741541e+12,-6.473827e+06,1.055059e+08,5.475604e+07


In [17]:
# Wide-form pivot: (run, budget_type, location, component) columns
wide = long_df.pivot_table(
    index="datetime",
    columns=["run", "budget_type", "location", "component"],
    values="value")
print("wide shape:", wide.shape)
wide.iloc[:3, :3]

wide shape: (3653, 333)


run                      baseline                                          
budget_type                    GW                                          
location        ENTIRE MODEL AREA                                          
component   Beginning Storage (+) Boundary Inflow (+) Cumulative Subsidence
datetime                                                                   
1990-10-02           1.640838e+12        1.656614e+07          7.501907e+07
1990-10-03           1.640645e+12        1.565458e+07          8.157517e+07
1990-10-04           1.640529e+12        1.563629e+07          8.269750e+07

In [18]:
# Targeted subset load: filter runs/types/locations/dates at read time
subset = collect_budgets(
    runs=runs, budget_files=budget_files,
    budget_types=["GW"], locations=["Region1 (SR1)"],
    begin_date="1995-01-01", end_date="2000-12-31")
print(f"GW / Region1 / 1995-2000: {len(subset):,} rows")

GW / Region1 / 1995-2000: 35,717 rows


## Recap

- `read_budget_hdf`, `read_head_hdf`, `read_hydrograph_hdf`,
  `read_zbudget_hdf` cover the HDF outputs
- `read_hydrograph_out`, `read_final_state_out`, `read_budget_text`,
  `read_flow_out`, `read_velocity_out` cover the text outputs
- `collect_budgets` builds the one long-form frame you want for
  multi-run analysis — notebook 06 adds scenario runs to it